Ten skrypt tworzy system zarządzania opieką nad samochodem przy użyciu LlamaIndex, wczytując dane o problemach, częściach, kosztach i harmonogramach z plików JSON do baz wektorowych. Następnie definiuje funkcje (narzędzia) do diagnostyki, szacowania kosztów i planowania przeglądów, które są udostępniane agentowi LLM do odpowiadania na zapytania użytkowników.

# Setup

In [1]:
!pip install -qU llama-parse llama-index-llms-openai llama-index-embeddings-openai llama-index-vector-stores-lancedb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.6/263.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 9.1 MB/s eta 0:00:00


Konkretnie instaluje następujące biblioteki:

*   `llama-parse`: Biblioteka do parsowania różnych formatów danych (np. plików tekstowych, JSON).
*   `llama-index-llms-openai`: Integracja `LlamaIndex` z modelami językowymi OpenAI (takimi jak GPT-3, GPT-4).  `LlamaIndex` to framework ułatwiający budowanie aplikacji wykorzystujących duże modele językowe.
*   `llama-index-embeddings-openai`: Integracja `LlamaIndex` z usługą tworzenia embeddingów OpenAI. Embeddingi reprezentują tekst w postaci wektorów, co pozwala na wykonywanie operacji semantycznych (np. wyszukiwanie podobnych dokumentów).
*   `llama-index-vector-stores-lancedb`:  Integracja `LlamaIndex` z bazą danych wektorowych LanceDB. LanceDB służy do przechowywania i szybkiego wyszukiwania embeddingów.


In [3]:
import json
import os
from datetime import datetime, timedelta

from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex,
)
from llama_index.core.agent import AgentRunner, FunctionCallingAgentWorker
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.tools import FunctionTool
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.lancedb import LanceDBVectorStore

try:
    from google.colab import userdata
except ModuleNotFoundError:
    print("This is not google colab env.")

import pandas as pd

This is not google colab env.


Ten kod importuje szereg bibliotek i modułów Pythona, które są niezbędne do działania aplikacji wykorzystującej `LlamaIndex` oraz integracji z OpenAI i LanceDB. Oto szczegółowe wyjaśnienie każdego importu:

Z biblioteki `llama_index.core` importowane są:

*   `Document`: Klasa reprezentująca pojedynczy dokument tekstowy.
*   `Settings`:  Klasa do konfigurowania globalnych ustawień `LlamaIndex`.
*   `StorageContext`: Klasa zarządzająca przechowywaniem i pobieraniem danych (np. embeddingów, indeksów).
*   `VectorStoreIndex`:  Klasa reprezentująca indeks wektorowy, który umożliwia szybkie wyszukiwanie podobnych dokumentów.

Z `llama_index.core.agent` importowane są:

*   `AgentRunner`: Klasa do uruchamiania agentów (systemów wykorzystujących LLM do podejmowania decyzji).
*   `FunctionCallingAgentWorker`:  Klasa implementująca agenta, który może wywoływać funkcje na podstawie zapytań.

Z `llama_index.core.node_parser` importowany jest:

*   `SentenceSplitter`: Klasa do dzielenia tekstu na zdania (nody).

Z `llama_index.core.retrievers` importowany jest:

*   `VectorIndexRetriever`:  Klasa odpowiedzialna za pobieranie dokumentów z indeksu wektorowego na podstawie zapytania.

Z `llama_index.core.tools` importowane są:

*   `FunctionTool`: Klasa reprezentująca narzędzie (funkcję), które agent może używać.
*   `ToolOutput`:  Klasa reprezentująca wynik działania narzędzia.

Z `llama_index.embeddings.openai` importowany jest:

*   `OpenAIEmbedding`: Klasa do generowania embeddingów tekstowych przy użyciu API OpenAI.

Z `llama_index.llms.openai` importowany jest:

*   `OpenAI`:  Klasa reprezentująca model językowy OpenAI (np. GPT-3, GPT-4).

Z `llama_index.vector_stores.lancedb` importowany jest:

*   `LanceDBVectorStore`: Klasa do przechowywania i wyszukiwania wektorów w bazie danych LanceDB.

In [9]:
class CFG:
    model1 = "gpt-4o-mini"
    model2 = "text-embedding-3-large"
    temperature = 0.1
    chunksize = 1024
    datadir = "./content/"

In [ ]:
# Using OpenAI API for embeddings/llms
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")


In [ ]:
llm = OpenAI(model=CFG.model1)
Settings.llm = llm

embed_model = OpenAIEmbedding(model=CFG.model2)
Settings.embed_model = embed_model

# Funkcje

In [ ]:
def load_and_index_document_from_file(
    file_path: str, vector_store: LanceDBVectorStore
) -> VectorStoreIndex:
    """Load a document from a single file and index it."""
    with open(file_path, "r") as f:
        data = json.load(f)
        document = Document(text=json.dumps(data))

    parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
    nodes = parser.get_nodes_from_documents([document])
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    return VectorStoreIndex(nodes, storage_context=storage_context)


Ta funkcja ładuje dane z pliku JSON, dzieli je na mniejsze fragmenty (nody) i indeksuje w bazie danych wektorowej LanceDB. Pozwala to na efektywne wyszukiwanie informacji zawartych w pliku.

In [7]:
def create_retriever(index: VectorStoreIndex) -> VectorIndexRetriever:
    """Create a retriever from the index."""
    return index.as_retriever(similarity_top_k=5)

Ta funkcja tworzy obiekt retrievera z istniejącego indeksu wektorowego, który umożliwia wyszukiwanie podobnych dokumentów na podstawie zapytania użytkownika.

# Dane

In [ ]:
pd.read_json(CFG.datadir + "cars_models.json").head(3)

,name,description,metadata,steps
0,Brake Pad Replacement,A service to replace worn brake pads with new ...,"{'source': 'Automotive Manual A', 'difficulty'...","[Raise the vehicle and remove the wheel., Remo..."
1,Oil Change,A service to replace old engine oil with new o...,"{'source': 'Automotive Manual B', 'difficulty'...",[Raise the vehicle and locate the oil drain pl...
2,Tire Rotation,A service to rotate the tires to ensure even t...,"{'source': 'Automotive Manual C', 'difficulty'...","[Raise the vehicle and remove the wheels., Rot..."


In [ ]:
pd.read_json(CFG.datadir + "cost_estimates.json").head(3)

,name,description,metadata
0,Brake Pads,High-quality brake pads designed for optimal s...,"{'category': 'Brake System', 'brand': 'AutoMas..."
1,Engine Oil,Synthetic engine oil that provides superior pr...,"{'category': 'Lubricants', 'brand': 'EnginePro..."
2,Air Filter,Premium air filter to ensure clean air flow to...,"{'category': 'Air Intake System', 'brand': 'Fi..."


In [13]:
pd.read_json(CFG.datadir + "diagnostics.json").head(3)

,symptom,possible_causes,severity,recommended_action,related_problem
0,Squealing brakes,"[Worn brake pads, Glazed brake rotors, Lack of...",Medium,Inspect brake system and replace worn parts,Brake Pad Replacement
1,Engine overheating,"[Low coolant level, Faulty thermostat, Broken ...",High,Stop driving immediately and have the cooling ...,Coolant Flush
2,Poor engine performance,"[Dirty air filter, Faulty spark plugs, Clogged...",Medium,Inspect and replace necessary components,Air Filter Replacement


In [ ]:
pd.read_json(CFG.datadir + "maintenance.json").head(3)

,repair,average_cost,cost_range
0,Brake pad replacement,150,"{'min': 100, 'max': 300}"
1,Oil change,50,"{'min': 20, 'max': 100}"
2,Air Filter Replacement,30,"{'min': 15, 'max': 50}"


In [ ]:
pd.read_json(CFG.datadir + "parts.json").head(3)

,mileage,tasks,importance,estimated_time
0,30000,"[Oil and filter change, Tire rotation, Air fil...",Regular maintenance,2-3 hours
1,60000,"[Transmission fluid change, Spark plug replace...",Major service,4-6 hours
2,90000,"[Coolant flush, Battery replacement, Serpentin...",Major service,3-4 hours


In [10]:
pd.read_json(CFG.datadir + "problems.json").head(3)

,name,description,metadata,steps
0,Brake Pad Replacement,A service to replace worn brake pads with new ...,"{'source': 'Automotive Manual A', 'difficulty'...","[Raise the vehicle and remove the wheel., Remo..."
1,Oil Change,A service to replace old engine oil with new o...,"{'source': 'Automotive Manual B', 'difficulty'...",[Raise the vehicle and locate the oil drain pl...
2,Tire Rotation,A service to rotate the tires to ensure even t...,"{'source': 'Automotive Manual C', 'difficulty'...","[Raise the vehicle and remove the wheels., Rot..."


In [ ]:
problems_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="problems_table",
    mode="overwrite",
)

problems_index = load_and_index_document_from_file(
    CFG.datadir + "problems.json", problems_vector_store
)

cost_estimates_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="cost_estimates_table",
    mode="overwrite",
)

cost_estimates_index = load_and_index_document_from_file(
    CFG.datadir + "cost_estimates.json",
    cost_estimates_vector_store,
)

diagnostics_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="diagnostics_table",
    mode="overwrite",
)

diagnostics_index = load_and_index_document_from_file(
    CFG.datadir + "diagnostics.json",
    diagnostics_vector_store,
)

parts_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="parts_table",
    mode="overwrite",
)

parts_index = load_and_index_document_from_file(
    CFG.datadir + "parts.json", parts_vector_store
)

maintenance_schedules_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="maintenance_schedules_table",
    mode="overwrite",
)

maintenance_schedules_index = load_and_index_document_from_file(
    CFG.datadir + "maintenance.json",
    maintenance_schedules_vector_store,
)


cars_vector_store = LanceDBVectorStore(
    uri="./lancedb",
    table_name="car_maintenance_table",
    mode="overwrite",
)

cars_index = load_and_index_document_from_file(
    CFG.datadir + "cars_models.json", cars_vector_store
)

Funkcja `load_and_index_document_from_file` wykonuje następujące kroki: ładuje dane z pliku JSON, dzieli je na fragmenty, generuje embeddingi dla tych fragmentów i przechowuje je w bazie danych LanceDB, tworząc indeks wektorowy. Wynikiem jest obiekt `problems_index`, który reprezentuje ten indeks wektorowy.

In [ ]:
problems_retriever = create_retriever(problems_index)
parts_retriever = create_retriever(parts_index)
car_details_retriever = create_retriever(cars_index)
diagnostics_retriever = create_retriever(diagnostics_index)
cost_estimates_retriever = create_retriever(cost_estimates_index)
maintenance_schedules_retriever = create_retriever(maintenance_schedules_index)

Ten kod tworzy po jednym retrieverze dla każdego z indeksów wektorowych, które zostały wcześniej utworzone. Każdy retriever będzie służył do wyszukiwania informacji w odpowiadającym mu zbiorze danych.

*   `problems_retriever = create_retriever(problems_index)`: Tworzy retrievera dla indeksu `problems_index`, który zawiera dane dotyczące problemów z pojazdami.
*   `parts_retriever = create_retriever(parts_index)`: Tworzy retrievera dla indeksu `parts_index`, który zawiera dane dotyczące części zamiennych.
*   `cars_retriever = create_retriever(cars_index)`: Tworzy retrievera dla indeksu `cars_index`, który zawiera dane dotyczące samochodów.
*   `diagnostics_retriever = create_retriever(diagnostics_index)`: Tworzy retrievera dla indeksu `diagnostics_index`, który zawiera dane dotyczące diagnostyki pojazdów.
*   `cost_estimates_retriever = create_retriever(cost_estimates_index)`: Tworzy retrievera dla indeksu `cost_estimates_index`, który zawiera dane dotyczące szacunków kosztów napraw.
*   `maintenance_schedules_retriever = create_retriever(maintenance_schedules_index)`: Tworzy retrievera dla indeksu `maintenance_schedules_index`, który zawiera dane dotyczące harmonogramów przeglądów i konserwacji.

In [20]:
# test retriever
query = "My brake pad isn't working , what's the cost for the solution?"
query_engine = cost_estimates_index.as_query_engine()
response = query_engine.query(query)
results = cost_estimates_retriever.retrieve(query)

# Print the response summary
print(f"Response: {response}")

# Print only relevant information from results
for result in results:
    print(f"Result - Node ID: {result.node_id}")
    print(f"Relevant Text: {result.text[:150]}...")
    print(f"Score: {result.score:.3f}")

Response: The cost for brake pad replacement typically averages around $150, with a range from $100 to $300.
Result - Node ID: 83b9c3f0-6f19-4601-9880-3f6a2f440402
Relevant Text: [{"repair": "Brake pad replacement", "average_cost": 150, "cost_range": {"min": 100, "max": 300}}, {"repair": "Oil change", "average_cost": 50, "cost_...
Score: 0.302
Result - Node ID: 7ad900bf-7d94-4d03-9c86-874e6ee46a4d
Relevant Text: "max": 600}}, {"repair": "Fuel Pump Replacement", "average_cost": 500, "cost_range": {"min": 400, "max": 700}}, {"repair": "AC Compressor Replacement"...
Score: 0.281


# Agent setup

In [ ]:
def retrieve_problems(query: str) -> str:
    """Searches the problem catalog to find relevant automotive problems for the query."""
    docs = problems_retriever.retrieve(query)
    return str([doc.text[:200] for doc in docs])


def retrieve_parts(query: str) -> str:
    """Searches the parts catalog to find relevant parts for the query."""
    docs = parts_retriever.retrieve(query)
    return str([doc.text[:200] for doc in docs])


def retrieve_car_details(make: str, model: str, year: int) -> str:
    """Retrieves the make, model, and year of the car."""
    docs = car_details_retriever.retrieve(make, model, year)
    return str([doc.text[:200] for doc in docs])


def diagnose_car_problem(symptoms: str) -> str:
    """Uses the diagnostics database to find potential causes for given symptoms."""
    docs = diagnostics_retriever.retrieve(symptoms)
    return str([doc.text[:200] for doc in docs])


def estimate_repair_cost(problem: str) -> str:
    """Provides a cost estimate for a given car problem or repair."""
    docs = cost_estimates_retriever.retrieve(problem)
    return str([doc.text[:200] for doc in docs])


def get_maintenance_schedule(mileage: int) -> str:
    """Retrieves the recommended maintenance schedule based on mileage."""
    docs = maintenance_schedules_retriever.retrieve(str(mileage))
    return str([doc.text[:200] for doc in docs])

Podsumowując, te funkcje stanowią interfejs do wyszukiwania informacji w różnych katalogach danych dotyczących samochodów za pomocą wcześniej utworzonych retrieverów. Każda funkcja przyjmuje odpowiednie parametry zapytania i zwraca tekst zawierający fragmenty najbardziej trafnych dokumentów znalezionych przez retrievera. Ograniczenie długości tekstu do 200 znaków ma na celu zapewnienie zwięzłych wyników wyszukiwania.

In [ ]:
def comprehensive_diagnosis(symptoms: str) -> str:
    """
    Provides a comprehensive diagnosis including possible causes, estimated costs, and required parts.

    Args:
        symptoms: A string describing the car's symptoms.

    Returns:
        A string with a comprehensive diagnosis report.
    """
    # Use existing tools
    possible_causes = diagnose_car_problem(symptoms)

    # Extract the most likely cause (this is a simplification)
    likely_cause = possible_causes[0] if possible_causes else "Unknown issue"

    estimated_cost = estimate_repair_cost(likely_cause)
    required_parts = retrieve_parts(likely_cause)

    report = "Comprehensive Diagnosis Report:\n\n"
    report += f"Symptoms: {symptoms}\n\n"
    report += f"Possible Causes:\n{possible_causes}\n\n"
    report += f"Most Likely Cause: {likely_cause}\n\n"
    report += f"Estimated Cost:\n{estimated_cost}\n\n"
    report += f"Required Parts:\n{required_parts}\n\n"
    report += "Please note that this is an initial diagnosis. For accurate results, please consult with our professional mechanic."

    return report


def get_car_model_info(
    mileage: int, car_make: str, car_model: str, car_year: int
) -> dict:
    """Retrieve car model information from cars_models.json."""
    with open(CFG.datadir + "cars_models.json", "r") as file:
        car_models = json.load(file)

    for car in car_models:
        if (
            car["car_make"].lower() == car_make.lower()
            and car["car_model"].lower() == car_model.lower()
            and car["car_year"] == car_year
        ):
            return car
    return {}


def retrieve_car_details(make: str, model: str, year: int) -> str:
    """Retrieves the make, model, and year of the car and return the common issues if any."""
    car_details = get_car_model_info(
        0, make, model, year
    )  # Using 0 for mileage to get general details
    if car_details:
        return f"{year} {make} {model} - Common Issues: {', '.join(car_details['common_issues'])}"
    return f"{year} {make} {model} - No common issues found."


In [ ]:
def plan_maintenance(mileage: int, car_make: str, car_model: str, car_year: int) -> str:
    """
    Creates a comprehensive maintenance plan based on the car's mileage and details.

    Args:
        mileage: The current mileage of the car.
        car_make: The make of the car.
        car_model: The model of the car.
        car_year: The year the car was manufactured.

    Returns:
        A string with a comprehensive maintenance plan.
    """
    car_details = retrieve_car_details(car_make, car_model, car_year)
    car_model_info = get_car_model_info(mileage, car_make, car_model, car_year)

    plan = f"Maintenance Plan for {car_year} {car_make} {car_model} at {mileage} miles:\n\n"
    plan += f"Car Details: {car_details}\n\n"

    if car_model_info:
        plan += "Common Issues:\n"
        for issue in car_model_info["common_issues"]:
            plan += f"- {issue}\n"

        plan += f"\nEstimated Time: {car_model_info['estimated_time']}\n\n"
    else:
        plan += (
            "No specific maintenance tasks found for this car model and mileage.\n\n"
        )

    plan += "Please consult with our certified mechanic for a more personalized maintenance plan."

    return plan


def create_calendar_invite(
    event_type: str, car_details: str, duration: int = 60
) -> str:
    """
    Simulates creating a calendar invite for a car maintenance or repair event.

    Args:
        event_type: The type of event (e.g., "Oil Change", "Brake Inspection").
        car_details: Details of the car (make, model, year).
        duration: Duration of the event in minutes (default is 60).

    Returns:
        A string describing the calendar invite.
    """
    # Simulate scheduling the event for next week
    event_date = datetime.now() + timedelta(days=7)
    event_time = event_date.replace(hour=10, minute=0, second=0, microsecond=0)

    invite = "Calendar Invite Created:\n\n"
    invite += f"Event: {event_type} for {car_details}\n"
    invite += f"Date: {event_time.strftime('%Y-%m-%d')}\n"
    invite += f"Time: {event_time.strftime('%I:%M %p')}\n"
    invite += f"Duration: {duration} minutes\n"
    invite += "Location: Your Trusted Auto Shop, 123 Main St, Bengaluru, India\n\n"

    return invite


def coordinate_car_care(
    query: str, car_make: str, car_model: str, car_year: int, mileage: int
) -> str:
    """
    Coordinates overall car care by integrating diagnosis, maintenance planning, and scheduling.

    Args:
        query: The user's query or description of the issue.
        car_make: The make of the car.
        car_model: The model of the car.
        car_year: The year the car was manufactured.
        mileage: The current mileage of the car.

    Returns:
        A string with a comprehensive car care plan.
    """
    car_details = retrieve_car_details(car_make, car_model, car_year)

    # Check if it's a problem or routine maintenance
    if "problem" in query.lower() or "issue" in query.lower():
        diagnosis = comprehensive_diagnosis(query)
        plan = f"Based on your query, here's a diagnosis:\n\n{diagnosis}\n\n"

        # Extract the most likely cause (this is a simplification)
        likely_cause = diagnosis.split("Most Likely Cause:")[1].split("\n")[0].strip()

        # Create a calendar invite for repair
        invite = create_calendar_invite(f"Repair: {likely_cause}", car_details)
        plan += f"I've prepared a calendar invite for the repair:\n\n{invite}\n\n"
    else:
        maintenance_plan = plan_maintenance(mileage, car_make, car_model, car_year)
        plan = f"Here's your maintenance plan:\n\n{maintenance_plan}\n\n"

        # Create a calendar invite for the next maintenance task
        next_task = maintenance_plan.split("Task:")[1].split("\n")[0].strip()
        invite = create_calendar_invite(f"Maintenance: {next_task}", car_details)
        plan += f"I've prepared a calendar invite for your next maintenance task:\n\n{invite}\n\n"

    plan += "Remember to consult with a professional mechanic for personalized advice and service."

    return plan


In [24]:
retrieve_problems_tool = FunctionTool.from_defaults(fn=retrieve_problems)
retrieve_parts_tool = FunctionTool.from_defaults(fn=retrieve_parts)
diagnostic_tool = FunctionTool.from_defaults(fn=diagnose_car_problem)
cost_estimator_tool = FunctionTool.from_defaults(fn=estimate_repair_cost)
maintenance_schedule_tool = FunctionTool.from_defaults(fn=get_maintenance_schedule)
comprehensive_diagnostic_tool = FunctionTool.from_defaults(fn=comprehensive_diagnosis)
maintenance_planner_tool = FunctionTool.from_defaults(fn=plan_maintenance)
calendar_invite_tool = FunctionTool.from_defaults(fn=create_calendar_invite)
car_care_coordinator_tool = FunctionTool.from_defaults(fn=coordinate_car_care)
retrieve_car_details_tool = FunctionTool.from_defaults(fn=retrieve_car_details)


Ten kod tworzy obiekty `FunctionTool` dla każdej z wcześniej zdefiniowanych funkcji, co umożliwia ich wykorzystanie jako narzędzi przez agenta w frameworku `LlamaIndex`.

In [ ]:
tools = [
    retrieve_problems_tool,
    retrieve_parts_tool,
    diagnostic_tool,
    cost_estimator_tool,
    maintenance_schedule_tool,
    comprehensive_diagnostic_tool,
    maintenance_planner_tool,
    calendar_invite_tool,
    car_care_coordinator_tool,
    retrieve_car_details_tool,
]


In [ ]:
agent_worker = FunctionCallingAgentWorker.from_tools(tools, llm=llm, verbose=True)
agent = AgentRunner(agent_worker)

# Run

In [27]:
response = agent.chat(
    "My car has 60,000 miles on it. What maintenance should I be doing now, and how much will it cost?"
)

Added user message to memory: My car has 60,000 miles on it. What maintenance should I be doing now, and how much will it cost?
=== LLM Response ===
To provide you with the best maintenance plan and cost estimate, I need to know the make, model, and year of your car. Could you please provide that information?


In [28]:
response = agent.chat(
    "I have a honda accord of 2017 model and it's mileage is 30000 right now, what are some common issues?"
)

Added user message to memory: I have a honda accord of 2017 model and it's mileage is 30000 right now, what are some common issues?
=== Calling Function ===
Calling function: retrieve_car_details with args: {"make": "Honda", "model": "Accord", "year": 2017}
=== Function Output ===
2017 Honda Accord - Common Issues: Engine misfires, Electrical issues
=== LLM Response ===
The common issues reported for the 2017 Honda Accord include:

1. **Engine Misfires**: This can occur due to various reasons such as faulty spark plugs, ignition coils, or fuel injectors.
2. **Electrical Issues**: Problems with the electrical system, including battery or alternator failures, can arise.

If you have any specific symptoms or concerns, please let me know, and I can assist you further!


In [29]:
response = agent.chat(
    "Can you help me with these issues, I want to do some maintenance and what's the cost for all of this services? for parts and all which will be required"
)

Added user message to memory: Can you help me with these issues, I want to do some maintenance and what's the cost for all of this services? for parts and all which will be required
=== Calling Function ===
Calling function: comprehensive_diagnosis with args: {"symptoms": "Engine misfires, Electrical issues"}
=== Function Output ===
Comprehensive Diagnosis Report:

Symptoms: Engine misfires, Electrical issues

Possible Causes:
['[{"symptom": "Squealing brakes", "possible_causes": ["Worn brake pads", "Glazed brake rotors", "Lack of lubrication on brake hardware"], "severity": "Medium", "recommended_action": "Inspect brake syst', '"recommended_action": "Inspect and replace necessary components", "related_problem": "Power Steering Fluid Change"}, {"symptom": "Fuel odor", "possible_causes": ["Leaking fuel line", "Faulty fuel pres']

Most Likely Cause: [

Estimated Cost:
['"max": 600}}, {"repair": "Fuel Pump Replacement", "average_cost": 500, "cost_range": {"min": 400, "max": 700}}, {"repai